# Tokopedia iPhone 17 Scraper

Web scraping produk iPhone 17 dari Tokopedia menggunakan Playwright dengan stealth plugin.

## Step 1: Install Dependencies

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
# Install required packages
%pip install playwright playwright-stealth pandas fake-useragent nest-asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# Install Playwright browser (run once)
!playwright install chromium

## Step 2: Import Libraries & Load URLs

In [3]:
import time
import random
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from playwright.sync_api import sync_playwright

# Manual stealth function (sync version)
def apply_stealth(page):
    """Apply stealth settings to avoid bot detection."""
    page.add_init_script("""
        // Override webdriver property
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        });
        
        // Override plugins
        Object.defineProperty(navigator, 'plugins', {
            get: () => [1, 2, 3, 4, 5]
        });
        
        // Override languages
        Object.defineProperty(navigator, 'languages', {
            get: () => ['id-ID', 'id', 'en-US', 'en']
        });
        
        // Override chrome property
        window.chrome = {
            runtime: {}
        };
        
        // Override permissions
        const originalQuery = window.navigator.permissions.query;
        window.navigator.permissions.query = (parameters) => (
            parameters.name === 'notifications' ?
                Promise.resolve({ state: Notification.permission }) :
                originalQuery(parameters)
        );
    """)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [4]:
# URLs from link.txt (embedded directly)
all_urls = [
    "https://www.tokopedia.com/digitechmall/digitech-apple-iphone-17-256gb-512gb-id-sa-pa-a-garansi-resmi-1732763953374004851",
    "https://www.tokopedia.com/igoodsgadget/iphone-17-garansi-resmi-indonesia-1732924766502749859",
    "https://www.tokopedia.com/universestore/apple-iphone-17-garansi-resmi-apple-indonesia-1733338491722696145",
    "https://www.tokopedia.com/ponselnesia/live-apple-iphone-17-256gb-512gb-garansi-resmi-indonesia-1732837428295140519",
    "https://www.tokopedia.com/qstore-pedia/apple-iphone-17-256gb-512gb-garansi-resmi-apple-indonesia-1732783070905533525",
    "https://www.tokopedia.com/supergadget-729/apple-iphone-17-garansi-resmi-ibox-indonesia-1732581947732952388",
    "https://www.tokopedia.com/digitechmall/resmi-apple-iphone-17-pro-256gb-512gb-1tb-a19-pro-garansi-resmi-id-pa-sa-a-digitech-1732911270140610163",
    "https://www.tokopedia.com/ismile-indonesia/apple-iphone-17-pro-garansi-resmi-256gb-512gb-1tb-1733012623030978443",
    "https://www.tokopedia.com/agadget-10/asm-tp-apple-iphone-17-basic-256gb-512gb-garansi-resmi-1732794377943025222",
    "https://www.tokopedia.com/digitalisme/apple-iphone-17-pro-max-promax-iphone-17-pro-256gb-512gb-1tb-2tb-garansi-resmi-1732920330740597969",
    "https://www.tokopedia.com/putra-group/resmi-iphone-17-256gb-512gb-a19-chip-black-white-mist-blue-sage-lavender-1733168013654721536",
    "https://www.tokopedia.com/senz88/apple-iphone-17-pro-max-garansi-resmi-indonesia-1732836066545076177",
    "https://www.tokopedia.com/senz88/apple-iphone-17-pro-garansi-resmi-indonesia-1732952524298618833",
    "https://www.tokopedia.com/distriponsel/apple-iphone-17-pro-256gb-512gb-1tb-a19-pro-chip-blue-orange-silver-garansi-apple-1732842705254516134",
    "https://www.tokopedia.com/gadgetpoint/apple-iphone-17-256gb-512gb-garansi-resmi-ibox-gdn-1732866351908750766",
    "https://www.tokopedia.com/putra-group/resmi-iphone-17-pro-max-promax-256gb-512gb-1tb-a19-pro-chip-silver-cosmic-orange-deep-blue-1733168061590176768",
    "https://www.tokopedia.com/putra-group/resmi-iphone-17-pro-256gb-512gb-1tb-a19-pro-chip-silver-cosmic-orange-deep-blue-1733168019728335872",
    "https://www.tokopedia.com/ini-toko-budi-shop/apple-iphone-17-pro-max-256-512-1tb-garansi-resmi-ibox-tam-1732876298985244135",
    "https://www.tokopedia.com/agadget-10/asm-x-k-apple-iphone-17-basic-256gb-512gb-garansi-resmi-1733146320935749190",
    "https://www.tokopedia.com/studioponsel/apple-iphone-17-pro-chip-a19-pro-1tb-512gb-256gb-silver-cosmic-orange-deep-blue-garansi-resmi-1733035894408185650",
    "https://www.tokopedia.com/milano-cell/fs-milano-apple-iphone-17-256gb-garansi-resmi-1733680775158727703",
    "https://www.tokopedia.com/collinsofficial/apple-iphone-13-iphone-14-iphone-15-iphone-16-iphone-17-resmi-ibox-digimap-gdn-1733480299558110407"
]

print(f"📋 Total URLs loaded: {len(all_urls)}")
for i, url in enumerate(all_urls[:5], 1):
    print(f"  {i}. {url[:80]}...")
print(f"  ... dan {len(all_urls) - 5} URL lainnya")

📋 Total URLs loaded: 22
  1. https://www.tokopedia.com/digitechmall/digitech-apple-iphone-17-256gb-512gb-id-s...
  2. https://www.tokopedia.com/igoodsgadget/iphone-17-garansi-resmi-indonesia-1732924...
  3. https://www.tokopedia.com/universestore/apple-iphone-17-garansi-resmi-apple-indo...
  4. https://www.tokopedia.com/ponselnesia/live-apple-iphone-17-256gb-512gb-garansi-r...
  5. https://www.tokopedia.com/qstore-pedia/apple-iphone-17-256gb-512gb-garansi-resmi...
  ... dan 17 URL lainnya


## Step 3: Scraping Function

In [8]:
import subprocess
import json
import sys
import pandas as pd
from datetime import datetime

def run_scraper(urls: list, headless: bool = False) -> pd.DataFrame:
    """
    Run Playwright scraper in subprocess to avoid asyncio conflicts.
    Uses the same Python interpreter as the notebook kernel.
    """
    urls_json = json.dumps(urls)
    
    # Use the same Python executable as the notebook kernel
    python_exe = sys.executable
    print(f"📌 Using Python: {python_exe}")
    
    # Run scraper.py as subprocess
    result = subprocess.run(
        [python_exe, "scraper.py", urls_json, str(headless).lower()],
        capture_output=True,
        text=True,
        encoding='utf-8'
    )
    
    # Print progress messages (stderr)
    if result.stderr:
        print(result.stderr)
    
    # Parse JSON results (stdout)
    if result.stdout.strip():
        try:
            data = json.loads(result.stdout)
            return pd.DataFrame(data)
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
            print(f"Raw output: {result.stdout[:500]}")
            return pd.DataFrame()
    else:
        print("❌ No data returned from scraper")
        return pd.DataFrame()

print(f"✅ Scraper function defined!")
print(f"📌 Current Python: {sys.executable}")

✅ Scraper function defined!
📌 Current Python: c:\Users\narev\Documents\xai_lime_vs_shap\venv312\Scripts\python.exe


## Step 4: Test Scraping (3 URLs)

⚠️ **Browser akan terbuka secara otomatis.** Jangan tutup browser sampai scraping selesai.

In [11]:
# Test dengan 3 URL pertama
test_urls = all_urls[:3]
print(f"🧪 Testing dengan {len(test_urls)} URLs...")
print("⚠️ Browser Chromium akan terbuka. Jangan ditutup sampai selesai!\n")

# Jalankan scraping via subprocess
df_test = run_scraper(test_urls, headless=False)

print("\n" + "="*60)
print("📊 TEST RESULTS")
print("="*60)

# Debug info
print(f"DataFrame shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

if not df_test.empty and 'product_name' in df_test.columns:
    display(df_test[['product_name', 'price', 'store_name', 'rating', 'review_count', 'stock_status']])
else:
    print("No data scraped or missing columns")
    print(df_test)

🧪 Testing dengan 3 URLs...
⚠️ Browser Chromium akan terbuka. Jangan ditutup sampai selesai!

📌 Using Python: c:\Users\narev\Documents\xai_lime_vs_shap\venv312\Scripts\python.exe
🚀 Launching browser (headless=False)...

🔍 [1/3] https://www.tokopedia.com/digitechmall/digitech-apple-iphone...
  ❌ Failed: Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://www.tokopedia.com/digitechmall/digitech-apple-iphone-17-256gb-512gb-id-sa-pa-a-garansi-resmi-1732763953374004851", waiting until "networkidle"


🔍 [2/3] https://www.tokopedia.com/igoodsgadget/iphone-17-garansi-res...
  ❌ Failed: Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://www.tokopedia.com/igoodsgadget/iphone-17-garansi-resmi-indonesia-1732924766502749859", waiting until "networkidle"


🔍 [3/3] https://www.tokopedia.com/universestore/apple-iphone-17-gara...
  ❌ Failed: Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://www.tokopedia.com/universestore/apple-iphone-

,product_name,price,store_name,rating,review_count,stock_status
0,None,None,None,None,None,None
1,None,None,None,None,None,None
2,None,None,None,None,None,None


In [7]:
# Check data completeness
print("📋 Data Completeness Check:")
for col in ['product_name', 'price', 'store_name', 'rating', 'review_count']:
    filled = df_test[col].notna().sum()
    total = len(df_test)
    pct = (filled / total) * 100
    status = "✅" if pct == 100 else "⚠️" if pct >= 50 else "❌"
    print(f"  {status} {col}: {filled}/{total} ({pct:.0f}%)")

📋 Data Completeness Check:


KeyError: 'product_name'

## Step 5: Full Scraping (All 20 URLs)

⚠️ **Jalankan cell ini setelah test berhasil.** Estimasi waktu: ~40-60 menit.

In [ ]:
# Full scraping - semua URLs
print(f"🚀 Starting full scrape of {len(all_urls)} URLs...")
print("⏱️ Estimasi waktu: 40-60 menit")
print("⚠️ Browser akan terbuka. Jangan ditutup!\n")

df_full = run_scraper(all_urls, headless=False)

print("\n" + "="*60)
print("📊 FULL SCRAPING RESULTS")
print("="*60)
df_full

## Step 6: Review Statistics & Export

In [ ]:
# Review statistics
print("📊 REVIEW STATISTICS")
print("="*50)

# Use df_full if available, else df_test
df = df_full if 'df_full' in dir() else df_test

# Total reviews
total_reviews = df['review_count'].sum()
print(f"📝 Total Reviews: {total_reviews:,}")

# Average rating
avg_rating = df['rating'].mean()
print(f"⭐ Average Rating: {avg_rating:.2f}")

# Product with most reviews
if df['review_count'].notna().any():
    max_review_idx = df['review_count'].idxmax()
    max_review_product = df.loc[max_review_idx]
    print(f"\n🏆 Most Reviewed Product:")
    print(f"   {max_review_product['product_name'][:60]}...")
    print(f"   Reviews: {max_review_product['review_count']:,}")

# Product with least reviews  
if df['review_count'].notna().any():
    min_review_idx = df['review_count'].idxmin()
    min_review_product = df.loc[min_review_idx]
    print(f"\n📉 Least Reviewed Product:")
    print(f"   {min_review_product['product_name'][:60]}...")
    print(f"   Reviews: {min_review_product['review_count']:,}")

# Data completeness summary
print(f"\n✅ Data Completeness:")
for col in ['product_name', 'price', 'store_name', 'rating', 'review_count']:
    filled = df[col].notna().sum()
    print(f"   {col}: {filled}/{len(df)}")

In [ ]:
# Export to CSV
output_file = f"iphone17_tokopedia_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"💾 Data exported to: {output_file}")
print(f"📁 Total records: {len(df)}")

# Also save as Excel for better viewing
excel_file = output_file.replace('.csv', '.xlsx')
df.to_excel(excel_file, index=False)
print(f"📁 Excel file: {excel_file}")